# Normalização de colunas no banco

Funções auxiliares para aplicar fatores de normalização diretamente em tabelas PostgreSQL usando SQLAlchemy.

In [92]:
from sqlalchemy import Engine, MetaData, Table, cast, func, update
from sqlalchemy.sql.sqltypes import BigInteger, DOUBLE_PRECISION



import os
from pathlib import Path


from sql_utils import drop_table, build_postgres_engine


project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

engine = build_postgres_engine(
        "localhost",
        int(os.environ.get("POSTGRES_PORT", 5432)),
        os.environ["POSTGRES_DB"],
        os.environ["POSTGRES_USER"],
        os.environ["POSTGRES_PASSWORD"],
    )
    
chunck_size = 100_000 # Se estiver estourando mt ram, abaixe esse numero

# Normalização das tabelas

Para reduzir problemas de instabilidade estatística associados a unidades territoriais com população reduzida, os municípios analisados foram classificados em três faixas populacionais: municípios pequenos, com menos de 20 mil habitantes; municípios médios, com população entre 20 mil e 50 mil habitantes; e municípios grandes, com mais de 50 mil habitantes. A partir dessa classificação, optou-se por excluir da análise principal os municípios pequenos, isto é, aqueles com população inferior a 20 mil habitantes.

Essa decisão metodológica não decorre de um ponto de corte universal estabelecido pelo IBGE para todos os indicadores municipais, mas de uma opção operacional voltada à redução do ruído estatístico em análises comparativas cidade a cidade. Em unidades populacionais muito pequenas, pequenas variações absolutas podem produzir grandes oscilações relativas em taxas, proporções e indicadores per capita. Assim, mesmo alterações numéricas reduzidas podem distorcer comparações entre municípios, especialmente quando os indicadores analisados dependem de estimativas amostrais, projeções populacionais ou razões calculadas sobre denominadores pequenos.

O próprio IBGE destaca que estimativas provenientes de amostras estão sujeitas a erro amostral e que sua precisão pode ser avaliada pelo coeficiente de variação (CV). O CV é definido como uma medida relativa de precisão, obtida pela razão entre o erro-padrão e o valor estimado do indicador. Segundo o IBGE, quanto menor o coeficiente de variação, maior tende a ser a precisão da estimativa; de modo geral, quanto maior a ordem de grandeza do total estimado, maior a precisão e menor o CV.

Além disso, o desenho amostral do Censo Demográfico diferencia os municípios conforme seu porte populacional. O IBGE informa que, no Censo 2010, foram utilizadas frações amostrais distintas segundo faixas de população municipal: municípios com mais de 8 mil até 20 mil habitantes tiveram fração amostral de 20%, enquanto municípios com mais de 20 mil até 500 mil habitantes tiveram fração de 10%; para o Censo 2022, foi seguida a mesma lógica do Censo 2010. Embora essa divisão não represente um critério oficial de “confiabilidade mínima”, ela evidencia que o porte populacional é considerado pelo IBGE como elemento relevante no desenho das estimativas amostrais.

Dessa forma, o corte de 20 mil habitantes foi adotado como critério metodológico de robustez, buscando limitar a influência de municípios com maior instabilidade relativa nos indicadores analisados. Os municípios entre 20 mil e 50 mil habitantes foram mantidos como municípios médios, enquanto aqueles com mais de 50 mil habitantes foram classificados como grandes. Essa classificação permite preservar a comparação entre diferentes portes municipais, ao mesmo tempo em que reduz a presença de observações potencialmente mais sensíveis a altos coeficientes de variação e a flutuações decorrentes de pequenos denominadores populacionais.

Como referência para a aplicação desse corte populacional, serão utilizados os dados do Censo Demográfico 2022, armazenados na tabela raw.AMP_Demografico, especificamente na coluna quantidade_de_moradores. Assim, durante o processo de tratamento e normalização dos dados, as tabelas copiadas para o schema norm_por_100k já terão removidas as linhas referentes aos municípios classificados como pequenos, isto é, aqueles com quantidade_de_moradores inferior a 20 mil habitantes. Com isso, as análises posteriores realizadas sobre as tabelas normalizadas considerarão apenas municípios médios e grandes, conforme a classificação populacional adotada neste trabalho.

A normalização será realizada em contagem por 10 mil habitantes

fonte (isso é im SLIDE precisa achar a tabela da fonte orginal): https://agenciadenoticias.ibge.gov.br/media/com_mediaibge/arquivos/b48f07c18b14687fa95692972d6a92d4.pdf

In [93]:
from sqlalchemy import Column, MetaData, Table, insert, inspect, null, select, String, delete, text
from sqlalchemy.schema import CreateSchema
from sqlalchemy.sql.type_api import TypeEngine

from sql_utils import normalize_table_columns_by_factor

source_schema = "raw"
target_schema_name = "norm_por_1k"

def copy_table_between_schemas(
    engine: Engine,
    source_schema_name: str,
    source_table_name: str,
    target_schema_name: str,
    target_table_name: str | None = None,
    column_name_mapping: dict[str, str] | None = None,
    extra_columns: dict[str, TypeEngine] | None = None,
) -> None:
    target_table_name = target_table_name or source_table_name
    column_name_mapping = column_name_mapping or {}
    extra_columns = extra_columns or {}

    inspector = inspect(engine)
    if inspector.has_table(target_table_name, schema=target_schema_name):
        raise ValueError(f"Target table already exists: {target_schema_name}.{target_table_name}")

    source_metadata = MetaData(schema=source_schema_name)
    source_table = Table(source_table_name, source_metadata, autoload_with=engine)

    target_metadata = MetaData(schema=target_schema_name)
    target_columns = [
        Column(
            column_name_mapping.get(column.name, column.name),
            column.type,
            primary_key=column.primary_key,
            nullable=column.nullable,
        )
        for column in source_table.columns
    ]
    target_columns.extend(
        Column(column_name, column_type, nullable=True)
        for column_name, column_type in extra_columns.items()
    )
    target_table = Table(target_table_name, target_metadata, *target_columns)

    source_columns = list(source_table.columns)
    selected_columns = [*source_columns, *[null() for _ in extra_columns]]
    target_column_names = [column.name for column in target_table.columns]

    copy_statement = insert(target_table).from_select(
        target_column_names,
        select(*selected_columns),
    )

    with engine.begin() as connection:
        connection.execute(CreateSchema(target_schema_name, if_not_exists=True))
        target_table.create(connection)
        connection.execute(copy_statement)


def delete_missing_municipality_rows(
    engine: Engine,
    schema_name: str,
    table_name: str,
    territory_code_column: str,
    reference_table_name: str,
    reference_code_column: str,
) -> None:
    metadata = MetaData(schema=schema_name)
    table = Table(table_name, metadata, autoload_with=engine)
    reference_table = Table(reference_table_name, metadata, autoload_with=engine)

    territory_code = cast(table.c[territory_code_column], String)
    reference_code = cast(reference_table.c[reference_code_column], String)

    statement = delete(table).where(
        func.length(territory_code) > 2,
        territory_code.not_in(select(reference_code)),
    )

    with engine.begin() as connection:
        result = connection.execute(statement)

    print(f"Deleted rows: {result.rowcount}")

def delete_rows_below_threshold(
    engine: Engine,
    schema_name: str,
    table_name: str,
    column_name: str,
    threshold: int | float,
) -> None:
    metadata = MetaData(schema=schema_name)
    table = Table(table_name, metadata, autoload_with=engine)

    statement = delete(table).where(table.c[column_name] < threshold)

    with engine.begin() as connection:
        result = connection.execute(statement)

    print(f"Deleted rows: {result.rowcount}")

def rename_table_column(
    engine: Engine,
    schema_name: str,
    table_name: str,
    old_column_name: str,
    new_column_name: str,
) -> None:
    preparer = engine.dialect.identifier_preparer
    qualified_table_name = f"{preparer.quote_schema(schema_name)}.{preparer.quote(table_name)}"
    old_column = preparer.quote(old_column_name)
    new_column = preparer.quote(new_column_name)

    statement = text(
        f"ALTER TABLE {qualified_table_name} RENAME COLUMN {old_column} TO {new_column}"
    )

    with engine.begin() as connection:
        connection.execute(statement)

    print(f"Renamed column: {schema_name}.{table_name}.{old_column_name} -> {new_column_name}")

    


In [94]:
from typing import TypeAlias

from sqlalchemy.sql.elements import BinaryExpression, ColumnElement

ColumnRef: TypeAlias = tuple[str, str, str]
JoinPair: TypeAlias = tuple[ColumnRef, ColumnRef]


def divide_columns_and_update(
    engine: Engine,
    numerator: ColumnRef,
    denominator: ColumnRef,
    target: ColumnRef,
    join_pairs: list[JoinPair] | None = None,
    round_result: bool = False,
    cast_result_to: TypeEngine | None = None,
) -> None:
    """
    Atualiza uma coluna com o resultado da divisão entre duas colunas.

    Exemplo:
        target = numerator / denominator

    As colunas podem estar:
    - na mesma tabela;
    - em tabelas diferentes;
    - e a coluna target pode ser uma das colunas usadas na operação.

    ColumnRef:
        (schema_name, table_name, column_name)

    JoinPair:
        ((schema_a, table_a, column_a), (schema_b, table_b, column_b))
    """

    join_pairs = join_pairs or []

    metadata = MetaData()

    all_column_refs: list[ColumnRef] = [
        numerator,
        denominator,
        target,
    ]

    for left_ref, right_ref in join_pairs:
        all_column_refs.append(left_ref)
        all_column_refs.append(right_ref)

    table_refs = {
        (schema_name, table_name)
        for schema_name, table_name, _ in all_column_refs
    }

    tables: dict[tuple[str, str], Table] = {
        (schema_name, table_name): Table(
            table_name,
            metadata,
            schema=schema_name,
            autoload_with=engine,
        )
        for schema_name, table_name in table_refs
    }

    def get_column(column_ref: ColumnRef):
        schema_name, table_name, column_name = column_ref
        return tables[(schema_name, table_name)].c[column_name]

    numerator_col = get_column(numerator)
    denominator_col = get_column(denominator)
    target_col = get_column(target)

    target_schema, target_table_name, _ = target
    target_table = tables[(target_schema, target_table_name)]

    expression = numerator_col / func.nullif(denominator_col, 0)

    if round_result:
        expression = func.round(expression)

    if cast_result_to is not None:
        expression = cast(expression, cast_result_to)

    where_conditions: list[ColumnElement[bool]] = [
        cast(get_column(left_ref), BigInteger) == get_column(right_ref)
        for left_ref, right_ref in join_pairs
    ]

    stmt = (
        update(target_table)
        .values({target_col.name: expression})
    )

    if where_conditions:
        stmt = stmt.where(*where_conditions)

    with engine.begin() as connection:
        connection.execute(stmt)

# AMP_Demografico

In [95]:

source_table = "APM_Demografico"

target_table_name = "APM_Demografico"

copy_table_between_schemas(engine, source_schema, source_table, target_schema_name, target_table_name)



In [96]:
target_table_name = "APM_Demografico"

metadata = MetaData(schema=target_schema_name)

table = Table(
    target_table_name,
    metadata,
    autoload_with=engine,
)

code_text = cast(table.c.CD_MUN, String)
numeric_column_names = [
    column.name
    for column in table.columns
    if column.name not in {"id", "CD_MUN", "NM_MUN"}
]

with engine.begin() as connection:
    connection.execute(delete(table).where(func.length(code_text) <= 2))

    next_id = connection.scalar(
        select(
            func.coalesce(
                func.max(cast(table.c.id, BigInteger)),
                0,
            ) + 1
        )
    )

    brasil_values = {
        "id": next_id,
        "CD_MUN": 1,
        "NM_MUN": None,
    }

    for column_name in numeric_column_names:
        brasil_values[column_name] = connection.scalar(
            select(func.coalesce(func.sum(table.c[column_name]), 0))
            .where(func.length(code_text) > 2)
        )

    connection.execute(insert(table).values(brasil_values))
    next_id += 1

    state_codes = connection.scalars(
        select(distinct(cast(func.substr(code_text, 1, 2), BigInteger)))
        .where(func.length(code_text) > 2)
        .order_by(cast(func.substr(code_text, 1, 2), BigInteger))
    ).all()

    for state_code in state_codes:
        state_values = {
            "id": next_id,
            "CD_MUN": state_code,
            "NM_MUN": None,
        }

        for column_name in numeric_column_names:
            state_values[column_name] = connection.scalar(
                select(func.coalesce(func.sum(table.c[column_name]), 0))
                .where(
                    func.length(code_text) > 2,
                    cast(func.substr(code_text, 1, 2), BigInteger) == state_code,
                )
            )

        connection.execute(insert(table).values(state_values))
        next_id += 1


In [97]:
def preview_rows_below_threshold_cut(
    engine: Engine,
    schema_name: str,
    table_name: str,
    column_name: str,
    threshold: int | float,
) -> None:
    metadata = MetaData(schema=schema_name)
    table = Table(table_name, metadata, autoload_with=engine)

    total_statement = select(func.count()).select_from(table)
    removed_statement = select(func.count()).select_from(table).where(table.c[column_name] < threshold)

    with engine.connect() as connection:
        total_rows = connection.execute(total_statement).scalar_one()
        removed_rows = connection.execute(removed_statement).scalar_one()

    remaining_rows = total_rows - removed_rows

    print(f"Total rows: {total_rows}")
    print(f"Rows below threshold: {removed_rows}")
    print(f"Rows after cut: {remaining_rows}")

source_table = "APM_Demografico"

column_name = "quantidade_de_moradores"
threshold = 20_000

print("cidades com mais de 20 mil habitantes: ")
preview_rows_below_threshold_cut(
    engine,
    source_schema,
    source_table,
    column_name,
    threshold,
)

threshold = 50_000
print("cidades com mais de 50 mil habitantes: ")
preview_rows_below_threshold_cut(
    engine,
    source_schema,
    source_table,
    column_name,
    threshold,
)


cidades com mais de 20 mil habitantes: 
Total rows: 5570
Rows below threshold: 3863
Rows after cut: 1707
cidades com mais de 50 mil habitantes: 
Total rows: 5570
Rows below threshold: 4917
Rows after cut: 653


In [98]:
column_name = "quantidade_de_moradores"
threshold = 20_000

delete_rows_below_threshold(
    engine,
    target_schema_name,
    target_table_name,
    column_name,
    threshold,
)


Deleted rows: 3863


In [117]:
from sql_utils import print_table_head


source_table = "APM_Demografico"

print_table_head(engine, target_schema_name, target_table_name, rows=100)

      id cod_territorio                 territorio populacao        0_4      10_14      15_19      20_24     25_29     30_34     35_39     40_44     45_49     50_54     55_59        5_9     60_64     65_69     70_74     75_79     80_84    85_89    90_94    95_99  100_mais
      11              1                     Brasil    homens  31.899867  34.521573  36.124883  38.345397 37.655000 37.209836 38.641736 38.413292 32.331440 29.691660 26.754845  34.613081 22.737939 17.713385 12.911379  8.184107  4.985406 2.437031 0.959417 0.248413  0.052182
      12              1                     Brasil  mulheres  30.821094  32.988553  34.845825  38.008961 38.715452 39.177371 41.199600 40.931301 35.006645 32.504626 30.359161  33.264731 26.355214 21.169755 16.010861 10.809515  7.233246 4.124937 1.902572 0.567032  0.134497
      13              1                     Brasil     total  62.720961  67.510126  70.970708  76.354358 76.370452 76.387207 79.841336 79.344594 67.338085 62.196287 57.114006  67.87

# PIB_Geral

In [101]:
target_table_name = "Pib_Geral"

maping = {
    "Código do Município": "Cod",
    "Nome do Município": "Nome",
    """Produto Interno Bruto per capita, 
a preços correntes
(R$ 1,00""": "PIB_per_capita",
    """Produto Interno Bruto, 
a preços correntes
(R$ 1.000)""":
    "PIB",
}

ext_columns = {
    "populacao_projetada": BigInteger(),
    "populacao_proj_por_1000": DOUBLE_PRECISION()
}

copy_table_between_schemas(
    engine, 
    "raw", 
    "Pib_municipal", 
    target_schema_name, 
    target_table_name, 
    extra_columns=ext_columns
)

In [102]:
rename_table_column(
    engine,
    target_schema_name,
    target_table_name,
    """Produto Interno Bruto, 
a preços correntes
(R$ 1.000)""",
    "PIB",
)

rename_table_column(
    engine,
    target_schema_name,
    target_table_name,
    """Produto Interno Bruto per capita, 
a preços correntes
(R$ 1,00""",
    "PIB_per_capita",
)


rename_table_column(
    engine,
    target_schema_name,
    target_table_name,
    "Código do Município",
    "Cod",
)

rename_table_column(
    engine,
    target_schema_name,
    target_table_name,
    "Nome do Município",
    "Nome",
)

Renamed column: norm_por_1k.Pib_Geral.Produto Interno Bruto, 
a preços correntes
(R$ 1.000) -> PIB
Renamed column: norm_por_1k.Pib_Geral.Produto Interno Bruto per capita, 
a preços correntes
(R$ 1,00 -> PIB_per_capita
Renamed column: norm_por_1k.Pib_Geral.Código do Município -> Cod
Renamed column: norm_por_1k.Pib_Geral.Nome do Município -> Nome


## calculando os dados Brasil

In [105]:
from sqlalchemy import insert, distinct

target_table_name = "Pib_Geral"

metadata = MetaData(schema=target_schema_name)

table = Table(
    target_table_name,
    metadata,
    autoload_with=engine
)

with engine.begin() as connection:

    next_id = connection.scalar(
        select(
            func.coalesce(
                func.max(cast(table.c.id, BigInteger)),
                0,
            ) + 1
        )
        .where(table.c.id.op("~")("^[0-9]+$"))
    )


    anos = connection.scalars(
        select(distinct(table.c.Ano))
        .where(table.c.Ano.is_not(None))
        .order_by(table.c.Ano)
    ).all()

    for ano in anos:
        SUM_PIB = connection.scalar(
            select(
                func.coalesce(func.sum(table.c.PIB), 0)
            )
            .where(
                table.c.Ano == ano,
                table.c.Nome != "Brasil"
            )
        )

        SUM_PIB_per_capita = connection.scalar(
            select(
                func.coalesce(func.sum(table.c.PIB_per_capita), 0)
            )
            .where(
                table.c.Ano == ano,
                table.c.Nome != "Brasil"
            )
        )

        connection.execute(
            insert(table).values(
                {
                    "id": str(next_id),
                    "Ano": ano,
                    "Código da Grande Região": None,
                    "Nome da Grande Região": None,
                    "Código da Unidade da Federação": 1,
                    "Sigla da Unidade da Federação": "BR",
                    "Nome da Unidade da Federação": "Brasil",
                    "Cod": 1,
                    "Nome": "Brasil",
                    "Região Metropolitana": None,
                    "Código da Mesorregião": None,
                    "Nome da Mesorregião": None,
                    "Código da Microrregião": None,
                    "Nome da Microrregião": None,
                    "Código da Região Geográfica Imediata": None,
                    "Nome da Região Geográfica Imediata": None,
                    "Município da Região Geográfica Imediata": None,
                    "Código da Região Geográfica Intermediária": None,
                    "Nome da Região Geográfica Intermediária": None,
                    "Município da Região Geográfica Intermediária": None,
                    "Código Concentração Urbana": None,
                    "Nome Concentração Urbana": None,
                    "Tipo Concentração Urbana": None,
                    "Código Arranjo Populacional": None,
                    "Nome Arranjo Populacional": None,
                    "Hierarquia Urbana": None,
                    "Hierarquia Urbana (principais categorias)": None,
                    "Código da Região Rural": None,
                    "Nome da Região Rural": None,
                    "Região rural (segundo classificação do núcleo)": None,
                    "Cidade-Região de São Paulo": None,
                    "PIB": SUM_PIB,
                    "PIB_per_capita": SUM_PIB_per_capita,
                    "Atividade com maior valor adicionado bruto": None,
                    "Atividade com segundo maior valor adicionado bruto": None,
                    "Atividade com terceiro maior valor adicionado bruto": None,
                    "populacao_projetada": None,
                    "populacao_proj_por_1000": None
                }
            )
        )

        next_id += 1

## Calculando dados UF

In [106]:
from sqlalchemy import insert, distinct

target_table_name = "Pib_Geral"

metadata = MetaData(schema=target_schema_name)

table = Table(
    target_table_name,
    metadata,
    autoload_with=engine
)

with engine.begin() as connection:

    next_id = connection.scalar(
        select(
            func.coalesce(
                func.max(cast(table.c.id, BigInteger)),
                0,
            ) + 1
        )
        .where(table.c.id.op("~")("^[0-9]+$"))
    )


    anos = connection.scalars(
        select(distinct(table.c.Ano))
        .where(table.c.Ano.is_not(None))
        .order_by(table.c.Ano)
    ).all()

    ufs = connection.scalars(
        select(distinct(table.c["Código da Unidade da Federação"]))
        .where(table.c["Código da Unidade da Federação"].is_not(None))
        .order_by(table.c["Código da Unidade da Federação"])
    ).all()

    for ano in anos:

        for uf in ufs:
            SUM_PIB = connection.scalar(
                select(
                    func.coalesce(func.sum(table.c.PIB), 0)
                )
                .where(
                    table.c.Ano == ano,
                    table.c.Nome != "Brasil",
                    table.c["Código da Unidade da Federação"] == uf
                )
            )

            SUM_PIB_per_capita = connection.scalar(
                select(
                    func.coalesce(func.sum(table.c.PIB_per_capita), 0)
                )
                .where(
                    table.c.Ano == ano,
                    table.c.Nome != "Brasil",
                    table.c["Código da Unidade da Federação"] == uf
                )
            )

            UF_SIGLA = connection.scalar(
                select(table.c["Sigla da Unidade da Federação"])
                .where(
                    table.c.Ano == ano,
                    table.c["Código da Unidade da Federação"] == uf,
                    table.c.Nome != "Brasil",
                )
                .limit(1)
            )

            UF_NOME = connection.scalar(
                select(table.c["Nome da Unidade da Federação"])
                .where(
                    table.c.Ano == ano,
                    table.c["Código da Unidade da Federação"] == uf,
                    table.c.Nome != "Brasil",
                )
                .limit(1)
            )

            connection.execute(
                insert(table).values(
                    {
                        "id": str(next_id),
                        "Ano": ano,
                        "Código da Grande Região": None,
                        "Nome da Grande Região": None,
                        "Código da Unidade da Federação": uf,
                        "Sigla da Unidade da Federação": UF_SIGLA,
                        "Nome da Unidade da Federação": UF_NOME,
                        "Cod": uf,
                        "Nome": UF_NOME,
                        "Região Metropolitana": None,
                        "Código da Mesorregião": None,
                        "Nome da Mesorregião": None,
                        "Código da Microrregião": None,
                        "Nome da Microrregião": None,
                        "Código da Região Geográfica Imediata": None,
                        "Nome da Região Geográfica Imediata": None,
                        "Município da Região Geográfica Imediata": None,
                        "Código da Região Geográfica Intermediária": None,
                        "Nome da Região Geográfica Intermediária": None,
                        "Município da Região Geográfica Intermediária": None,
                        "Código Concentração Urbana": None,
                        "Nome Concentração Urbana": None,
                        "Tipo Concentração Urbana": None,
                        "Código Arranjo Populacional": None,
                        "Nome Arranjo Populacional": None,
                        "Hierarquia Urbana": None,
                        "Hierarquia Urbana (principais categorias)": None,
                        "Código da Região Rural": None,
                        "Nome da Região Rural": None,
                        "Região rural (segundo classificação do núcleo)": None,
                        "Cidade-Região de São Paulo": None,
                        "PIB": SUM_PIB,
                        "PIB_per_capita": SUM_PIB_per_capita,
                        "Atividade com maior valor adicionado bruto": None,
                        "Atividade com segundo maior valor adicionado bruto": None,
                        "Atividade com terceiro maior valor adicionado bruto": None,
                        "populacao_projetada": None,
                        "populacao_proj_por_1000": None
                    }
                )
            )

            next_id += 1

In [107]:
from sqlalchemy import MetaData, Table

target_table_name = "Pib_Geral"

metadata = MetaData(schema=target_schema_name)

table = Table(
    target_table_name,
    metadata,
    autoload_with=engine,
)

print(table.columns.keys())

['id', 'Ano', 'Código da Grande Região', 'Nome da Grande Região', 'Código da Unidade da Federação', 'Sigla da Unidade da Federação', 'Nome da Unidade da Federação', 'Cod', 'Nome', 'Região Metropolitana', 'Código da Mesorregião', 'Nome da Mesorregião', 'Código da Microrregião', 'Nome da Microrregião', 'Código da Região Geográfica Imediata', 'Nome da Região Geográfica Imediata', 'Município da Região Geográfica Imediata', 'Código da Região Geográfica Intermediária', 'Nome da Região Geográfica Intermediária', 'Município da Região Geográfica Intermediária', 'Código Concentração Urbana', 'Nome Concentração Urbana', 'Tipo Concentração Urbana', 'Código Arranjo Populacional', 'Nome Arranjo Populacional', 'Hierarquia Urbana', 'Hierarquia Urbana (principais categorias)', 'Código da Região Rural', 'Nome da Região Rural', 'Região rural (segundo classificação do núcleo)', 'Cidade-Região de São Paulo', 'PIB', 'PIB_per_capita', 'Atividade com maior valor adicionado bruto', 'Atividade com segundo maior

In [108]:
target_table_name = "Pib_Geral"

delete_missing_municipality_rows(
    engine,
    target_schema_name,
    target_table_name,
    "Cod",
    "APM_Demografico",
    "CD_MUN",
)

Deleted rows: 34767


In [109]:
target_table_name = "Pib_Geral"

normalize_table_columns_by_factor(
    engine,
    target_schema_name,
    target_table_name,
    ["PIB"],
    1000,
)

divide_columns_and_update(
    engine=engine,
    numerator=(target_schema_name, target_table_name, "PIB"),
    denominator=(target_schema_name, target_table_name, "PIB_per_capita"),
    target=(target_schema_name, target_table_name, "populacao_projetada"),
    round_result=True,
    cast_result_to=BigInteger(),
)

divide_columns_and_update(
    engine=engine,
    numerator=(target_schema_name, target_table_name, "PIB"),
    denominator=(target_schema_name, target_table_name, "PIB_per_capita"),
    target=(target_schema_name, target_table_name, "populacao_proj_por_1000"),
    round_result=True,
    cast_result_to=BigInteger(),
)

normalize_table_columns_by_factor(
    engine,
    target_schema_name,
    target_table_name,
    ["populacao_proj_por_1000"],
    1/1000,
)


In [120]:
from sql_utils import print_table_head

target_table_name = "Pib_Geral"

print_table_head(engine, target_schema_name, target_table_name, rows=100)

   id  Ano  Código da Grande Região Nome da Grande Região  Código da Unidade da Federação Sigla da Unidade da Federação Nome da Unidade da Federação     Cod                     Nome                                     Região Metropolitana  Código da Mesorregião    Nome da Mesorregião  Código da Microrregião  Nome da Microrregião  Código da Região Geográfica Imediata Nome da Região Geográfica Imediata Município da Região Geográfica Imediata  Código da Região Geográfica Intermediária Nome da Região Geográfica Intermediária Município da Região Geográfica Intermediária  Código Concentração Urbana Nome Concentração Urbana   Tipo Concentração Urbana  Código Arranjo Populacional                            Nome Arranjo Populacional     Hierarquia Urbana Hierarquia Urbana (principais categorias)  Código da Região Rural                                              Nome da Região Rural Região rural (segundo classificação do núcleo) Cidade-Região de São Paulo          PIB  PIB_per_capita          

In [121]:
target_table_name = "Pib_Geral"

metadata = MetaData(schema=target_schema_name)

table = Table(
    target_table_name,
    metadata,
    autoload_with=engine,
)

with engine.connect() as connection:
    result = connection.execute(
        select(table)
        .where(table.c["Código da Unidade da Federação"] == 1)
        .order_by(table.c.Ano, table.c.Cod)
    )

    for row in result.mappings():
        print(dict(row))


{'id': '77974', 'Ano': 2015, 'Código da Grande Região': None, 'Nome da Grande Região': None, 'Código da Unidade da Federação': 1, 'Sigla da Unidade da Federação': None, 'Nome da Unidade da Federação': None, 'Cod': 1, 'Nome': None, 'Região Metropolitana': None, 'Código da Mesorregião': None, 'Nome da Mesorregião': None, 'Código da Microrregião': None, 'Nome da Microrregião': None, 'Código da Região Geográfica Imediata': None, 'Nome da Região Geográfica Imediata': None, 'Município da Região Geográfica Imediata': None, 'Código da Região Geográfica Intermediária': None, 'Nome da Região Geográfica Intermediária': None, 'Município da Região Geográfica Intermediária': None, 'Código Concentração Urbana': None, 'Nome Concentração Urbana': None, 'Tipo Concentração Urbana': None, 'Código Arranjo Populacional': None, 'Nome Arranjo Populacional': None, 'Hierarquia Urbana': None, 'Hierarquia Urbana (principais categorias)': None, 'Código da Região Rural': None, 'Nome da Região Rural': None, 'Região 

# Dist Idade

In [110]:
source_table = "dist_idade"
target_table_name = "dist_idade"

copy_table_between_schemas(engine, source_schema, source_table, target_schema_name, target_table_name)

In [111]:
target_table_name = "dist_idade"

delete_missing_municipality_rows(
    engine,
    target_schema_name,
    target_table_name,
    "cod_territorio",
    "APM_Demografico",
    "CD_MUN",
)

Deleted rows: 11589


In [112]:
target_table_name = "dist_idade"

age_columns = [
    "0_4",
    "10_14",
    "15_19",
    "20_24",
    "25_29",
    "30_34",
    "35_39",
    "40_44",
    "45_49",
    "50_54",
    "55_59",
    "5_9",
    "60_64",
    "65_69",
    "70_74",
    "75_79",
    "80_84",
    "85_89",
    "90_94",
    "95_99",
    "100_mais",
]

normalize_table_columns_by_factor(
    engine,
    target_schema_name,
    target_table_name,
    age_columns,
    1000,
)


In [113]:
for column_name in age_columns:
    divide_columns_and_update(
        engine=engine,
        numerator=(target_schema_name, "dist_idade", column_name),
        denominator=(target_schema_name, "APM_Demografico", "quantidade_de_moradores"),
        target=(target_schema_name, "dist_idade", column_name),
        join_pairs=[
            (
                (target_schema_name, "dist_idade", "cod_territorio"),
                (target_schema_name, "APM_Demografico", "CD_MUN"),
            )
        ],
        round_result=False,
        cast_result_to=None,
    )

In [114]:
from sql_utils import print_table_head

target_table_name = "dist_idade"

print_table_head(engine, target_schema_name, target_table_name, rows=100)

      id cod_territorio                 territorio populacao        0_4      10_14      15_19      20_24     25_29     30_34     35_39     40_44     45_49     50_54     55_59        5_9     60_64     65_69     70_74     75_79     80_84    85_89    90_94    95_99  100_mais
      11              1                     Brasil    homens  31.899867  34.521573  36.124883  38.345397 37.655000 37.209836 38.641736 38.413292 32.331440 29.691660 26.754845  34.613081 22.737939 17.713385 12.911379  8.184107  4.985406 2.437031 0.959417 0.248413  0.052182
      12              1                     Brasil  mulheres  30.821094  32.988553  34.845825  38.008961 38.715452 39.177371 41.199600 40.931301 35.006645 32.504626 30.359161  33.264731 26.355214 21.169755 16.010861 10.809515  7.233246 4.124937 1.902572 0.567032  0.134497
      13              1                     Brasil     total  62.720961  67.510126  70.970708  76.354358 76.370452 76.387207 79.841336 79.344594 67.338085 62.196287 57.114006  67.87

In [116]:
from sqlalchemy import text


def drop_schema(
    engine: Engine,
    schema_name: str,
) -> None:
    preparer = engine.dialect.identifier_preparer
    quoted_schema = preparer.quote_schema(schema_name)

    with engine.begin() as connection:
        connection.execute(text(f"DROP SCHEMA IF EXISTS {quoted_schema} CASCADE"))

    inspector = inspect(engine)
    inspector.clear_cache()
    if inspector.has_schema(schema_name):
        raise RuntimeError(f"Schema still exists after DROP SCHEMA CASCADE: {schema_name}")

    print(f"Dropped schema and all objects inside it: {schema_name}")


# Para usar quando quiser recomeçar:
# drop_schema(engine, target_schema_name)


In [118]:
(31.899867+34.521573+36.124883+38.345397+37.655000+37.209836+38.641736+38.413292+32.331440+29.691660+26.754845+34.613081+22.737939+17.713385+12.911379+8.184107+4.985406+2.437031+0.959417+0.248413+0.052182)*1000

486431.869

In [119]:
(62.720961+67.510126+70.970708+76.354358+76.370452+76.387207+79.841336+79.344594+67.338085+62.196287+57.114006+67.877812+49.093153+38.883140+28.922240+18.993622+12.218652+6.561968+2.861988+0.815446+0.186679)*1000

1002562.82